# 19 — Modelos sequenciais: séries temporais e sentimentos

**EP · Data Science Knowledge Base | Deep Learning | Notebook 19/20**


**Pré-requisitos:** Python, NumPy, fundamentos de aprendizado supervisionado; JupyterLab; TensorFlow/Keras apenas nos laboratórios indicados.

**Escopo de validação:** os exemplos básicos podem ser reproduzidos sem TensorFlow; laboratórios opcionais precisam de validação local independente.

---

**Deep Learning · Capítulo de estudo independente**

> Material didático autoral · estudo independente · exemplos locais e laboratórios opcionais

**Referência curricular:** `material histórico de referência`. O material original foi preservado; este capítulo é uma reconstrução didática independente.

### Materiais complementares

- [Guia Técnico](./reference_materials/19_sequencias_series_sentimentos.pdf)
- [Cheat Sheet](./cheat_sheets/cheat_sheet_sequencias_series_sentimentos.pdf)


## Como estudar este capítulo

1. Leia a teoria e deduza as expressões apresentadas.
2. Execute as células do **núcleo obrigatório** em um kernel limpo.
3. Confira a interpretação e responda ao exercício antes de abrir a solução.
4. Se tiver TensorFlow e Keras compatíveis, execute o **laboratório opcional**.

**Pré-requisitos transversais:** Python (01), Estatística (03), preparação/validação de ML (05); aplicações textuais remetem à PLN (06). Não há necessidade de repetir integralmente esses cursos.

## Objetivos de aprendizagem

- Construir janelas temporais sem vazamento e diferenciar modelagem temporal de classificação de textos.
- Relacionar a técnica à sua função de perda, representação e protocolo de avaliação.
- Explicar limitações e formular um teste reprodutível.

## 1. Fundamentação conceitual

Janelas dependem de horizonte, contexto e data de corte. Em séries financeiras, retornos não são previsíveis apenas por suavidade aparente; comparar com baseline ingênuo e evitar extrapolação de evidência. Em sentimento, tokenização e embeddings são requisitos (ver área de Processamento de Linguagem Natural).

**Fonte histórica de apoio:** notas do curso original, consultadas apenas como referência de contexto; as explicações e exemplos desta edição são independentes.

**Questões críticas:** quais pressupostos são necessários? O que a função aprende? Qual métrica indica utilidade real? Que tipo de vazamento ou má interpretação invalidaria a conclusão?

## 2. Matemática essencial

**Expressão esquemática:** `X_t=(y_(t−k),…,y_(t−1))`

A expressão acima resume a relação central. Para a derivação, identifique dimensões, hipóteses e operações; não interprete notação compacta como uma implementação completa.

## 3. Experimento obrigatório — NumPy e/ou scikit-learn

Este experimento é propositalmente pequeno: **executa sem GPU, internet ou serviço externo**. Quando há treino de baseline, não interpretá-lo como evidência da qualidade de uma rede neural.

In [1]:
import numpy as np
serie=np.arange(25,dtype=float)
janela=4;horizonte=1
X=np.array([serie[i:i+janela] for i in range(len(serie)-janela-horizonte+1)])
y=np.array([serie[i+janela+horizonte-1] for i in range(len(X))])
corte=round(.75*len(X));print('Treino:',X[:corte].shape,'teste:',X[corte:].shape,'primeiro alvo:',y[0]);assert y[0]==4.

Treino: (16, 4) teste: (5, 4) primeiro alvo: 4.0


### Interpretação e limitações

Compare o resultado obtido com a formulação matemática. Confirme shapes, escala e hipótese de modelagem. **Um experimento sintético valida uma operação, não substitui avaliação em dados reais.** Ao treinar modelos, separe dados e ajuste transformações apenas no treino; preserve um conjunto final de teste.

## 4. Laboratório neural opcional — TensorFlow / Keras 3

Esta seção demonstra **APIs reais**, mas depende de versões compatíveis e pode exigir mais recursos que o núcleo. Não é necessária para executar o notebook básico. Não se deve confundir um teste de chamada de API com avaliação experimental de desempenho. Se o pacote TensorFlow não estiver disponível, será exibido um aviso de não execução.

In [2]:
# Laboratório opcional: execute após instalar e configurar TensorFlow/Keras compatíveis.
# Se TensorFlow estiver ausente, o exemplo é ignorado com aviso explícito.
import importlib.util
if importlib.util.find_spec('tensorflow') is None:
    print('Laboratório TensorFlow/Keras não executado: TensorFlow ausente neste kernel.')
else:
    import keras
    from keras import layers
    modelo=keras.Sequential([keras.Input(shape=(5,1)),layers.LSTM(8),layers.Dense(1)]);modelo.compile(optimizer='adam',loss='mse');modelo.summary()

I0000 00:00:1794602214.500145   31420 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 8)              │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │             9 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 329 (1.29 KB)

 Trainable params: 329 (1.29 KB)

 Non-trainable params: 0 (0.00 B)

## 5. Exercício de fixação

**Questão:** Por que embaralhar indiscriminadamente uma série temporal pode causar vazamento?

<details>
<summary>Confira a solução comentada</summary>

**Solução:** Porque amostras futuras podem influenciar treinamento e pré-processamento relativos ao passado.

**Verificação:** explique o resultado usando notação, shapes e consequências para generalização.
</details>

## 6. Erros comuns e boas práticas

- Confundir ajuste no treino com generalização para dados não vistos.
- Reportar somente uma métrica ou uma execução aleatória.
- Ignorar shape, tipo de variável e convenção do backend.
- Executar exemplos com pesos baixados sem registrar versão, licença e procedência.
- Confundir demonstração matemática com validação de um modelo de produção.

## 7. Resumo de consulta rápida

**Ideia central:** Janelas dependem de horizonte, contexto e data de corte.

**Expressão:** `X_t=(y_(t−k),…,y_(t−1))`.

**Lembrete:** começar por baseline; respeitar separação de dados; documentar decisões; avaliar limitações.

**Ponte com outros cursos:** ML I (05) para métricas e validação; PLN (06) para modelos de linguagem; projetos multidisciplinares ficam reservados à fase posterior.

**Fontes de atualização (consultar quando necessário):** [Keras 3](https://keras.io/guides/), [TensorFlow](https://www.tensorflow.org/guide), [scikit-learn](https://scikit-learn.org/stable/user_guide.html).

## Laboratório aplicado revisado — Previsão temporal com corte cronológico e baseline

Corte cronológico, janelas somente com passado e baseline ingênuo são necessários antes de avaliar LSTM para séries.

**Procedimento:** execute a próxima célula com o núcleo declarado em `requirements.txt`, confira os `assert` e interprete os resultados. O experimento é independente dos exemplos anteriores.

In [3]:
import numpy as np
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error
rng=np.random.default_rng(19);n=190;t=np.arange(n)
series=.03*t+np.sin(t/5)+rng.normal(0,.14,n)
k=6
X=np.array([series[i-k:i] for i in range(k,n)])
y=np.array([series[i] for i in range(k,n)])
cut=int(.75*len(X));Xt,Xe=X[:cut],X[cut:];yt,ye=y[:cut],y[cut:]
model=Ridge(alpha=1).fit(Xt,yt)
mae_model=mean_absolute_error(ye,model.predict(Xe))
mae_naive=mean_absolute_error(ye,Xe[:,-1])
print('MAE Ridge:',round(mae_model,3),'MAE último valor:',round(mae_naive,3),'teste posterior ao treino:',cut)
assert X.shape[0]==len(y)

MAE Ridge: 0.189 MAE último valor: 0.189 teste posterior ao treino: 138


### Exportação de resultados (local)

Execute esta célula após o experimento acima. Os arquivos são salvos em `outputs/` e não entram automaticamente no Git.

In [4]:
# Resultado da avaliação cronológica: nenhuma alteração na partição.
from pathlib import Path
import numpy as np
import pandas as pd

out=Path('outputs'); out.mkdir(exist_ok=True)
comparacao=pd.DataFrame({'indice_temporal':np.arange(cut+k,n),
                         'observado':np.asarray(ye,dtype=float),
                         'pred_ridge':np.asarray(model.predict(Xe),dtype=float),
                         'pred_naive_ultimo_valor':np.asarray(Xe[:,-1],dtype=float)})
assert len(comparacao)==len(ye)
caminho=out/'19_previsoes_series_temporais.csv'
comparacao.to_csv(caminho,index=False)
print('Exportado:',caminho)


Exportado: outputs/19_previsoes_series_temporais.csv


### Interpretação, limites e verificação

Separação temporal não substitui validação rolling-origin. A comparação com persistência é essencial; prever série sintética não comprova previsão financeira.

**Checklist:** conferir entradas e saídas, sementes, inexistência de uso dos dados de teste durante ajuste e distinção entre uma demonstração sintética e desempenho em aplicações reais.